<a href="https://colab.research.google.com/github/ebbettin/UCH_SRL/blob/main/Variant_ECLs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.9 MB/s  0:00:00


In [4]:
from Bio import SeqIO
import os
import zipfile

# Input and output folders
input_folder = "/content/input"
output_folder = "/content/fasta"

os.makedirs(output_folder, exist_ok=True)

# Select the FASTA file in the input folder
fasta_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".fasta", ".fa", ".faa", ".fna"))
]

if not fasta_files:
    raise FileNotFoundError(
        f"No FASTA files found in {input_folder}"
    )

# Use the first FASTA file found
fasta_file = os.path.join(input_folder, fasta_files[0])
print(f"Reading: {fasta_file}")

# Gene identifiers to search for
search_genes = [
    "TP0326",
    "TP0548",
    "TP0856",
    "TP0858",
    "TP0859",
    "TP0865"
]

# Read all sequences
records = list(SeqIO.parse(fasta_file, "fasta"))

# Extract each gene into its own FASTA file
for gene in search_genes:
    matched_records = [
        record for record in records
        if gene in record.description
    ]

    output_file = os.path.join(output_folder, f"{gene}.fasta")
    SeqIO.write(matched_records, output_file, "fasta")

    print(f"{gene}: {len(matched_records)} sequences saved")

print(f"\nFinished! Results saved in {output_folder}")

Reading: /content/input/2026-09-22_AA_variants.fasta
TP0326: 76 sequences saved
TP0548: 124 sequences saved
TP0856: 17 sequences saved
TP0858: 71 sequences saved
TP0859: 25 sequences saved
TP0865: 41 sequences saved

Finished! Results saved in /content/fasta


In [9]:
import requests
import os
import subprocess

# Get the latest official MUSCLE release
api_url = "https://api.github.com/repos/rcedgar/muscle/releases/latest"
response = requests.get(api_url)
response.raise_for_status()
release = response.json()

# Find the Linux x86-64 executable
assets = release["assets"]
matches = [
    asset for asset in assets
    if "linux" in asset["name"].lower()
    and ("intel64" in asset["name"].lower()
         or "linux64" in asset["name"].lower())
    and "aarch64" not in asset["name"].lower()
    and "arm64" not in asset["name"].lower()
]

if not matches:
    print("Available release files:")
    for asset in assets:
        print(asset["name"])
    raise RuntimeError("Could not identify the Linux x86-64 binary.")

asset = matches[0]
muscle_path = "/usr/local/bin/muscle"

# Download the actual executable
download = requests.get(asset["browser_download_url"])
download.raise_for_status()

with open(muscle_path, "wb") as f:
    f.write(download.content)

os.chmod(muscle_path, 0o755)

# Verify that MUSCLE runs
subprocess.run([muscle_path, "-version"], check=True)

# Input and output folders
input_folder = "/content/fasta"
output_folder = "/content/aligned"

os.makedirs(output_folder, exist_ok=True)

# Find FASTA files
fasta_files = []
for extension in ("*.fasta", "*.fa", "*.faa", "*.fna"):
    fasta_files.extend(glob.glob(os.path.join(input_folder, extension)))

if not fasta_files:
    raise FileNotFoundError(f"No FASTA files found in {input_folder}")

print(f"Found {len(fasta_files)} FASTA files.\n")

# Align each file using MUSCLE 5 super5
for fasta_file in sorted(fasta_files):

    filename = os.path.basename(fasta_file)
    gene_name = os.path.splitext(filename)[0]
    output_file = os.path.join(output_folder, f"{gene_name}_aligned.fasta")

    print(f"Aligning {filename}...")

    try:
        result = subprocess.run(
            [
                "muscle",
                "-super5", fasta_file,
                "-output", output_file
            ],
            capture_output=True,
            text=True,
            check=True
        )

        print(f"Saved: {output_file}")

    except subprocess.CalledProcessError as e:
        print(f"ERROR aligning {filename}:")
        print(e.stderr)

        if os.path.exists(output_file):
            os.remove(output_file)

print("\nAlignment process finished.")

Found 6 FASTA files.

Aligning TP0326.fasta...
Saved: /content/aligned/TP0326_aligned.fasta
Aligning TP0548.fasta...
Saved: /content/aligned/TP0548_aligned.fasta
Aligning TP0856.fasta...
Saved: /content/aligned/TP0856_aligned.fasta
Aligning TP0858.fasta...
Saved: /content/aligned/TP0858_aligned.fasta
Aligning TP0859.fasta...
Saved: /content/aligned/TP0859_aligned.fasta
Aligning TP0865.fasta...
Saved: /content/aligned/TP0865_aligned.fasta

Alignment process finished.


In [15]:
!pip install biopython pandas openpyxl

import os
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# =========================
# SETTINGS — adjust here
# =========================
excel_file = "/content/input/ECL_reference.xlsx"
alignment_folder = "/content/aligned"

extra_residues = 5  # Extra alignment columns on each side of the ECL

# Automatically name the output folder
output_folder = f"/content/ECL_sequences_{extra_residues}"
os.makedirs(output_folder, exist_ok=True)

# =========================
# READ ECL REFERENCE TABLE
# =========================
df = pd.read_excel(excel_file, header=None, dtype=str)
df = df.iloc[:, :3]
df.columns = ["Gene", "ECL", "Sequence"]

for col in ["Gene", "ECL", "Sequence"]:
    df[col] = df[col].fillna("").str.strip()
    df[col] = df[col].str.replace("\ufeff", "", regex=False)

df["Sequence"] = (
    df["Sequence"]
    .str.replace(r"\s+", "", regex=True)
    .str.upper()
)

# Keep valid ECL entries and ignore other rows
df = df[
    df["ECL"].str.match(r"^ECL\d+$", na=False)
    & (df["Gene"] != "")
    & (df["Sequence"] != "")
].copy()

print(f"Valid ECL entries: {len(df)}")
print(f"Extra alignment columns on each side: {extra_residues}")
print(f"Output folder: {output_folder}\n")

# =========================
# EXTRACT ECL REGIONS
# =========================
for _, row in df.iterrows():
    gene = row["Gene"]
    ecl = row["ECL"]
    reference_ecl = row["Sequence"]

    alignment_file = os.path.join(
        alignment_folder, f"{gene}_aligned.fasta"
    )

    if not os.path.isfile(alignment_file):
        print(f"SKIPPED {gene} {ecl}: alignment file not found")
        continue

    records = list(SeqIO.parse(alignment_file, "fasta"))

    if not records:
        print(f"SKIPPED {gene} {ecl}: alignment file is empty")
        continue

    # Confirm that all aligned sequences have the same length
    if len({len(record.seq) for record in records}) != 1:
        print(f"SKIPPED {gene} {ecl}: unequal alignment lengths")
        continue

    matches = []

    # Find the reference ECL sequence in the aligned proteins
    for record in records:
        aligned_seq = str(record.seq).upper()
        ungapped_seq = aligned_seq.replace("-", "").replace(".", "")

        start = ungapped_seq.find(reference_ecl)

        if start == -1:
            continue

        end = start + len(reference_ecl) - 1

        # Convert ungapped residue positions to alignment columns
        residue_count = 0
        aln_start = None
        aln_end = None

        for col, aa in enumerate(aligned_seq):
            if aa not in "-.":
                if residue_count == start:
                    aln_start = col
                if residue_count == end:
                    aln_end = col
                residue_count += 1

        if aln_start is not None and aln_end is not None:
            matches.append((record.id, aln_start, aln_end))

    if not matches:
        print(
            f"NOT FOUND: {gene} {ecl} — "
            "exact reference sequence not found"
        )
        continue

    # All matching sequences should map to the same alignment columns
    coordinates = {
        (start, end) for _, start, end in matches
    }

    if len(coordinates) != 1:
        print(
            f"AMBIGUOUS: {gene} {ecl} — "
            "matches map to different alignment regions"
        )
        continue

    ecl_start, ecl_end = next(iter(coordinates))

    # Extend the ECL boundaries by the requested number of columns
    aln_start = max(0, ecl_start - extra_residues)
    aln_end = min(
        len(records[0].seq) - 1,
        ecl_end + extra_residues
    )

    # Extract the same alignment columns from every sequence
    extracted_records = []

    for record in records:
        aligned_seq = str(record.seq).upper()
        region = aligned_seq[aln_start:aln_end + 1]

        extracted_records.append(
            SeqRecord(
                Seq(region),
                id=f"{record.id}_{ecl}",
                description=""
            )
        )

    # Save one FASTA file per gene and ECL
    output_file = os.path.join(
        output_folder, f"{gene}_{ecl}.fasta"
    )

    SeqIO.write(extracted_records, output_file, "fasta")

    print(
        f"SUCCESS: {gene} {ecl} — "
        f"{len(extracted_records)} sequences; "
        f"alignment columns {aln_start + 1}-{aln_end + 1}"
    )

print("\nFinished.")
print(f"Output folder: {output_folder}")


Valid ECL entries: 15
Extra alignment columns on each side: 5
Output folder: /content/ECL_sequences_5

SUCCESS: TP0326 ECL1 — 76 sequences; alignment columns 439-456
SUCCESS: TP0326 ECL2 — 76 sequences; alignment columns 473-485
SUCCESS: TP0326 ECL3 — 76 sequences; alignment columns 502-555
SUCCESS: TP0326 ECL4 — 76 sequences; alignment columns 575-606
SUCCESS: TP0326 ECL5 — 76 sequences; alignment columns 634-651
SUCCESS: TP0326 ECL6 — 76 sequences; alignment columns 678-731
SUCCESS: TP0326 ECL7 — 76 sequences; alignment columns 752-795
SUCCESS: TP0326 ECL8 — 76 sequences; alignment columns 811-837
SUCCESS: TP0548 ECL1 — 124 sequences; alignment columns 115-128
SUCCESS: TP0548 ECL2 — 124 sequences; alignment columns 141-184
SUCCESS: TP0548 ECL3 — 124 sequences; alignment columns 202-224
SUCCESS: TP0548 ECL4 — 124 sequences; alignment columns 246-272
SUCCESS: TP0548 ECL5 — 124 sequences; alignment columns 286-301
SUCCESS: TP0548 ECL6 — 124 sequences; alignment columns 315-328
SUCCESS: 